In [ ]:
"""
================================================================================
  SEISMIC AMPLITUDE SPECTRAL DENSITY (ASD) PLOTTER
================================================================================
  What this script does:
    - Reads seismic measurement data from a CSV file (recorded in the field)
    - Reads reference seismic data from LIGO Hanford Observatory (LHO)
    - Computes the Amplitude Spectral Density (ASD) for each channel
      (East, North, and Vertical/Z directions)
    - Plots the field data alongside the LIGO reference on a log-log graph

  How to use it:
    1. Set the FILE PATH to your CSV data file (Section 1)
    2. Adjust the PLOT SETTINGS if needed (Section 2)
    3. Run the script — three plots will appear (E, N, Z directions)

  Requirements:
    pandas, numpy, scipy, matplotlib
================================================================================
"""


###--------------------------------------------------------------------------------------------------------------------------###
'''------------------------------------------------- SECTION 1 — FILE PATH --------------------------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###


## Set this to the location of your seismic CSV data file.
## Only one path should be uncommented (no '#' at the start) at a time.

data_file_path = r"C:\Users\cacam\Documents\data_files\10-01-compare\seismo_test_cal_2024-10-01T22-07-09-752.csv"

# Other available field measurement files (comment/uncomment to switch):
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis20-cleanHarb-First-hills-by-road-to-AFB_2026-04-06T22-19-33-391.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis19-CleanHarb-first-hills-by-road-to-airbase_2026-04-06T22-08-23-898.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis18-CleanHarb-closer-with-gravel-trucks-NS-to-I80_2026-04-06T21-47-19-778.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis17-CleanHarb-closer-with-gravel-trucks-NS-to-I80_2026-04-06T21-39-22-735.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis16-DikeFirstWiden-NS-along-dike_2026-04-06T05-11-21-479.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis15-DikeFirstWiden-NS-along-dike_2026-04-06T05-06-59-405.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis14-CleanHarbor-closer-NS-to-i80_2026-04-06T03-22-50-930.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis13-CleanHarbor-closer-NS-to-i80_2026-04-06T03-18-25-309.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis12-15POLES-NS-to-i80_2026-04-06T02-35-21-127.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis11-15POLES-NS-to-i80_2026-04-06T02-28-51-709.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis10-15POLES-NS-to-i80_2026-04-06T02-21-58-225.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis09-Gypsonite-Junction-NS-I80_2026-04-06T01-48-27-253.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis08-Gypsonite-Junction-NS-I80_2026-04-06T01-41-23-216.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis07-close-i80-NS-to-i80_2026-04-06T00-32-21-479.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis06-clean-harb-close-noactivity-NS-along-road_2026-04-05T23-57-51-227.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis05-clean-harb-close-noactivity-NS-along-road_2026-04-05T23-53-21-314.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis03-clean-harb-close-noactivity_2026-04-05T23-30-57-948.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis02dUSKATOHV_2026-04-05T09-02-53-915.csv"
# data_file_path = r"/Users/rmss/Desktop/LIGO/CosmicExplorer/utah-first-measurements/Seis010hvVisibleonflats_2026-04-05T08-33-59-274.csv"


###--------------------------------------------------------------------------------------------------------------------------###
'''----------------------------------------------- SECTION 2 — PLOT SETTINGS ------------------------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###


## Adjust these to control how the output plots look and how the FFT is computed.


# Y-axis (amplitude) range in m/√Hz or m·s⁻¹/√Hz:
y_max = 10e-7                                            # Top of the plot    (higher = more noise visible)
y_min = 10e-12                                           # Bottom of the plot (lower = more detail at quiet levels)


# X-axis (frequency) range in Hz:
x_max = 100                                              # Highest frequency to show
x_min = 0.1                                              # Lowest frequency to show


# FFT (spectral analysis) settings:
fft_length = 8                                           # Length of each FFT window in seconds
                                                         #   Longer = better frequency resolution, less time averaging

overlap    = 50                                          # Overlap between consecutive FFT windows, in percent (0–99)


# Were the pre-amplifiers used during recording?
pre_amps = False                                         # Set to True if pre-amps were active


###--------------------------------------------------------------------------------------------------------------------------###
'''-------------------------------------------------- SECTION 3 — IMPORTS ---------------------------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###

print("Loading libraries...")
import warnings
import pandas as pd                    # For reading CSV files into tables
import numpy as np                     # For numerical calculations
from scipy import signal               # For the Welch PSD / spectral analysis
from matplotlib import pyplot as plt   # For creating plots
print("Libraries loaded.")

warnings.simplefilter('ignore')


###--------------------------------------------------------------------------------------------------------------------------###
'''------------------------------------------ SECTION 4 — READ THE FIELD DATA FILE ------------------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###


# --- Read metadata (first 5 lines) ---
metadata = {}
with open(data_file_path, 'r') as f:
    for _ in range(5):
        line = f.readline().strip()
        if ':' in line:
            key, value = line.split(':', 1)
            metadata[key.strip()] = value.strip()

print("\nFile metadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

sample_rate = float(metadata["Sample Rate"])


# --- Read the seismic data columns ---
raw_data = pd.read_csv(data_file_path, skiprows=5, delimiter=',')
raw_data.columns = ["Sample", "Time (s)", "Noise (V)",
                    "Channel E (V)", "Channel N (V)", "Channel Z (V)", "blank"]


# --- Apply calibration factor (Volts → meters per second) ---
if pre_amps:
    calibration = (0.0125e-1) / 21
else:
    calibration = 0.0125e-1

channel_east    = raw_data['Channel E (V)'] * calibration   # East  (X) direction
channel_north   = raw_data['Channel N (V)'] * calibration   # North (Y) direction
channel_z       = raw_data['Channel Z (V)'] * calibration   # Vertical (Z) direction


###--------------------------------------------------------------------------------------------------------------------------###
'''---------------------------------------- SECTION 5 — READ THE LIGO REFERENCE DATA ----------------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###


ligo_data = pd.read_csv(r"C:\Users\cacam\Documents\data_files\LHOSeismic.txt", delimiter=r"\s+")
ligo_data.columns = ["time", "x", "y", "z"]

ligo_east         = ligo_data["x"] * 1e-9
ligo_north        = ligo_data["y"] * 1e-9
ligo_z            = ligo_data["z"] * 1e-9
ligo_sample_rate  = 256


###--------------------------------------------------------------------------------------------------------------------------###
'''------------------------------------------- SECTION 6 — AXIS LABELS AND TITLES -------------------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###


plot_title_velocity     = "Seismic Velocity Data ASD"
plot_title_displacement = "Seismic Displacement Data ASD"

channel_titles          = {'e': "E Direction",
                           'n': "N Direction",
                           'z': "Z Direction"}

ylabel_velocity         = "Amplitude [ms⁻¹/√Hz]"
ylabel_displacement     = "Amplitude [m/√Hz]"


###--------------------------------------------------------------------------------------------------------------------------###
'''------------------------------------ SECTION 7 — ASD CALCULATION AND PLOTTING FUNCTION -----------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###


# Arguments:
#   data_vertical, data_north, data_east  — field data arrays (pass None to skip)
#   field_sample_rate                     — samples per second for field data
#   ref_sample_rate                       — samples per second for LIGO data
#   ref_east, ref_north, ref_vertical     — LIGO reference channel arrays
#   overlap                               — FFT window overlap in percent
#   fft_window_s                          — FFT window length in seconds
#   y_min, y_max                          — amplitude axis limits
#   x_min, x_max                          — frequency axis limits
#   plot_type                             — 'velocity' or 'displacement'
#   channel                               — 'all', 'east', 'north', or 'zed'


def compute_and_plot_asd(data_z, data_n, data_e, field_sample_rate, ref_sample_rate, ref_east, ref_north, ref_z, 
                         overlap, fft_window_s, y_min, y_max, x_min, x_max, plot_type, channel):
    
    
    # ------------------- Inner helper: compute ASD for a single data array ---------------------- #
    
    def welch_asd(data, sr):
        """
        Run Welch's method on `data` sampled at `sr` Hz.
        Returns (frequencies, amplitude_spectral_density).
        """
        data = np.asarray(data)   # Ensure it's a plain numpy array

        frequencies, power = signal.welch(
            data,
            sr,
            window   = 'hamming',
            nperseg  = int(sr * fft_window_s),                  ## nperseg:  number of samples per FFT window
            noverlap = int(round(sr * (overlap * 0.01) ) )      ## noverlap: number of samples shared between adjacent windows
        )
        
        # ASD = square root of PSD
        return frequencies, np.sqrt(power)

    # -------------------------- Map channel keys to their data arrays --------------------------- #
    
    field_channels = {'z': data_z,
                      'n': data_n,
                      'e': data_e}

    ligo_channels = {'z': ref_z,
                     'n': ref_north,
                     'e': ref_east}

    # ------------------------------------- Compute field ASDs ----------------------------------- #
    
    field_results = {}

    for key, data in field_channels.items():
        if data is None:
            continue   # Skip channels that weren't passed in

        freq, amp = welch_asd(data, field_sample_rate)

        if plot_type == 'displacement':
            # Convert velocity ASD → displacement ASD: divide by 2πf
            # (integrating in frequency domain)
            # Avoid division by zero at DC (freq = 0 Hz): set that bin to NaN
            disp = np.where(freq > 0, amp / (2 * np.pi * freq), np.nan)
        else:
            disp = None

        field_results[key] = {'frequency': freq,
                              'amp':       amp,
                              'disp':      disp}

    # -------------------------------- Compute LIGO reference ASDs ------------------------------- #
    
    ligo_results = {}

    for key, data in ligo_channels.items():
        if data is None:
            continue

        freq, amp = welch_asd(data, ref_sample_rate)

        if plot_type == 'displacement':
            disp = np.where(freq > 0, amp / (2 * np.pi * freq), np.nan)
        else:
            disp = None

        ligo_results[key] = {'frequency': freq,
                             'amp':       amp,
                             'disp':      disp}

    # ------------------------ Decide which channels to include in this plot ---------------------- #
    
    
    channel_name_to_key = {'east':  'e',
                          'north':  'n',
                          'zed':    'z'}

    if channel == 'all':
        channels_to_plot = ['z', 'n', 'e']
        plot_title = plot_title_velocity if plot_type == 'velocity' else plot_title_displacement
        
    else:
        key = channel_name_to_key[channel]
        channels_to_plot = [key]
        plot_title = channel_titles[key]
        

    # ---------------------------------- Color coding per channel --------------------------------- #
    
    line_colors = {'z': 'black',
                   'n': 'red',
                   'e': 'blue'}

    
    # --------------------------- Select y-axis label based on plot type -------------------------- #
    
    ylabel = ylabel_velocity if plot_type == 'velocity' else ylabel_displacement

    
    # ---------------------------------------- Draw the plot -------------------------------------- #
    
    plt.figure(figsize = (20, 8))
    plt.yscale('log')
    plt.xscale('log')

    for ch in channels_to_plot:
        if ch not in field_results:
            continue

        # Choose velocity or displacement data for y-axis
        if plot_type == 'velocity':
            y_field = field_results[ch]['amp']
            y_ligo  = ligo_results[ch]['amp']
        else:
            y_field = field_results[ch]['disp']
            y_ligo  = ligo_results[ch]['disp']

        f_field = field_results[ch]['frequency']
        f_ligo  = ligo_results[ch]['frequency']

        # Field data — solid line, channel color
        plt.plot(f_field, y_field,
                 color     = line_colors[ch],
                 linewidth = 1.75,
                 label     = channel_titles[ch])

        # LIGO reference — grey, semi-transparent
        plt.plot(f_ligo, y_ligo,
                 color     = 'dimgrey',
                 linewidth = 2,
                 alpha     = 0.5,
                 label     = f'LIGO {channel_titles[ch]}')

    # --------------------------------------- Axis formatting ------------------------------------- #
    
    ax = plt.gca()

    
    plt.title(plot_title, fontweight = 'bold', fontsize = 25)
    plt.title(f'FFT: {fft_window_s}s', fontsize = 18, loc = 'left', style = 'italic')
    plt.title(f'Overlap: {overlap}%', fontsize = 18, loc = 'right', style = 'italic')

    plt.xlabel('Frequency [Hz]', fontweight = 'bold', fontsize = 20)
    plt.ylabel(ylabel,           fontweight = 'bold', fontsize = 20)
    
    plt.yticks(fontsize = 20, fontweight = 'bold')
    plt.xticks(fontsize = 20, fontweight = 'bold')

    
    ax.tick_params(axis = 'both', which = 'minor', labelsize = 16)
    for label in ax.get_yticklabels(which = 'minor'):
        label.set_fontweight('bold')
    for label in ax.get_xticklabels(which = 'minor'):
        label.set_fontweight('bold')

    plt.ylim(y_min, y_max)
    plt.xlim(x_min, x_max)

    plt.legend(loc = 'lower left', fontsize = 16, ncol = 2)
    plt.grid(True, which = 'both', ls = '-')
    plt.tight_layout()
    plt.show()


###--------------------------------------------------------------------------------------------------------------------------###
'''------------------------------------------------ SECTION 8 — RUN THE PLOTS -----------------------------------------------'''
###--------------------------------------------------------------------------------------------------------------------------###

# Shared keyword arguments to avoid repeating them three times
common_args = dict(
    field_sample_rate = sample_rate,
    ref_sample_rate   = ligo_sample_rate,
    ref_east          = ligo_east,
    ref_north         = ligo_north,
    ref_z             = ligo_z,
    overlap           = overlap,
    fft_window_s      = fft_length,
    y_min             = y_min,
    y_max             = y_max,
    x_min             = x_min,
    x_max             = x_max,
    plot_type         = "displacement"
)

# ── All Channels (combined plot) — uncomment to enable ────────────────────────
# compute_and_plot_asd(channel_z, channel_north, channel_east,
#                      channel='all', **common_args)

# ── East Channel ──────────────────────────────────────────────────────────────
compute_and_plot_asd(None, None, channel_east,
                     channel = 'east', **common_args)

# ── North Channel ─────────────────────────────────────────────────────────────
compute_and_plot_asd(None, channel_north, None,
                     channel = 'north', **common_args)

# ── Vertical (Z) Channel ──────────────────────────────────────────────────────
compute_and_plot_asd(channel_z, None, None,
                     channel = 'zed', **common_args)